In [122]:
import os
import pandas as pd

In [123]:
def parse_desc_target(filename: str):
    parsed= filename.split(sep='_')
    parsed = [string for string in parsed if string not in ['p', 'd', 'mean', 'log']]
    target = parsed.pop().replace('.csv', '')
    return '_'.join(parsed), target

def get_mean_files(directory):
    files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    files = [file for file in files if 'mean' in file]
    descs, targets = zip(*[parse_desc_target(file) for file in files])
    return files, descs, targets

def get_mean_at_max_train_size(df: pd.DataFrame):
    return df[df['Train_sizes'] == df['Train_sizes'].max()]

def files_to_vals(filepath:str, filenames: list[str]) -> list[pd.DataFrame]:
    dfs = []
    for file in filenames:
        df = pd.read_csv(f'{filepath}/{file}')
        df = df.dropna(axis=1)
        df = df[['Test_MAE', 'Test_MSE', 'Test_R2', 'Train_sizes']]
        dfs.append(get_mean_at_max_train_size(df).drop(columns='Train_sizes'))
    return dfs


In [124]:
datasets = ['Wang', 'GeckoQ', 'Li', 'Ferraz-Caetano']
dirs = [
    f'../data/{dataset}/KRR_output' \
    for dataset in datasets
]


In [125]:
descs_to_label = {
    'ATMOMACCS_ALT_v3' : 'ATMOMACCS v3 (oxygen)',
    'ATMOMACCS_v1' : 'ATMOMACCS v1',
    'ATMOMACCS_v2' : 'ATMOMACCS v2',
    'ATMOMACCS_v3' : 'ATMOMACCS v3',
    'ATMOMACCS_v4' : 'ATMOMACCS v4',
    'ATMOMACCS_DECIMAL_v4' : 'ATMOMACCS v5',
    'ATMO_DECIMAL_v4' : 'ATMO v5',
    'ATMO_v4' : 'ATMO v4',
    'TopFP_kwg' : 'TopFP',
    'TopFP_kwiomg' : 'TopFP',
    'TopFP_sat' : 'TopFP',
    'TopFP_tg' : 'TopFP',
    'TopFP_dvap' : 'TopFP',
    'MACCS' : 'MACCS',
}

In [126]:
all_dfs_dict = {}

for dataset, dir in zip(datasets, dirs):
    all_dfs = []
    files, descs, targets = get_mean_files(dir)
    dfs = files_to_vals(dir, files)
    
    for df, file, desc in zip(dfs, files, descs):
        df['descriptor'] = descs_to_label[desc]
        df = df[['descriptor', 'Test_MAE']]
        # assign target
        for target in set(targets):
            if target in file:
                df['target'] = target
                break  # stop at first match
        df = df.round(3)
        df = df.sort_values(by='descriptor')
        all_dfs.append(df)
    
    all_dfs_dict[dataset] = pd.concat(all_dfs, ignore_index=True).sort_values(by='target')


C:\Users\kuuli\AppData\Local\Temp\ipykernel_38252\3596896182.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['target'] = target
C:\Users\kuuli\AppData\Local\Temp\ipykernel_38252\3596896182.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['target'] = target
C:\Users\kuuli\AppData\Local\Temp\ipykernel_38252\3596896182.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the 

In [ ]:
for key, val in all_dfs_dict.items():
    val = val[val['descriptor'] == 'ATMOMACCS v5']
    if key == 'Wang':
        for target in ['sat', 'kwg', 'kwiomg']:
            print(key, target)
            print(val[val['target'] == target].sort_values(by='descriptor') )
    elif key == 'GeckoQ':
        for target in ['sat']:
            print(key, target)
            print(val[val['target'] == target].sort_values(by='descriptor') )
    else:   
        print(key)
        print(val.sort_values(by='descriptor'))

Wang sat
   descriptor  Test_MAE target
20    ATMO v5     0.428    sat
Wang kwg
   descriptor  Test_MAE target
18    ATMO v5     0.609    kwg
Wang kwiomg
   descriptor  Test_MAE  target
19    ATMO v5     0.384  kwiomg
GeckoQ sat
   descriptor  Test_MAE target
13    ATMO v5     0.824    sat
Li
  descriptor  Test_MAE target
6    ATMO v5    22.175     tg
Ferraz-Caetano
  descriptor  Test_MAE target
6    ATMO v5     5.033   dvap
